# Task 1 — Baseline: Classical Lexical Retrieve-and-Extract
Dataset: TriviaQA (`rc.nocontext`) — Project 10

## 0. Overview

Classical pipeline: sample 300 TriviaQA (`rc.nocontext`) questions -> build a Wikipedia-paragraph
index from scratch -> BM25 retrieval -> heuristic (no-generation) extraction -> EM/F1/recall@k.

## 1. Setup, config, and fixed seeds

In [1]:
# Importing the libraries we need for data handling, retrieval, and progress bars
import os
import re
import json
import time
import random
import string
import urllib.request
import urllib.parse
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from datasets import load_dataset

import nltk

# Fixing the seed everywhere so the 300-question sample and every downstream
# result stays reproducible across runs (and across Tasks 2 & 3, which reuse
# this same seed)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Setting up the config constants used throughout this notebook
N_SAMPLE = 300          # number of TriviaQA validation questions sampled
TOP_K_RETRIEVE = 10     # retrieve enough passages to compute recall@10
K_LIST = [1, 3, 5, 10]  # recall@k values to report
TOP_K_FOR_ANSWER = 5    # spec: answer extraction looks at top-k=5 retrieved passages

# Setting up the data/results folders (created fresh if they don't already exist)
DATA_DIR = Path("../../23cs30011_project/data")
RESULTS_DIR = Path("../../23cs30011_project/results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Setting up the file paths used to persist the Wikipedia cache, the paragraph
# index, and the fixed question sample (so Tasks 2 & 3 can reuse them directly)
WIKI_CACHE_PATH = DATA_DIR / "wiki_pages_cache.json"
INDEX_PATH = DATA_DIR / "wiki_paragraph_index.jsonl"
SAMPLE_PATH = DATA_DIR / "triviaqa_sample_300.jsonl"

# Wikipedia asks for a descriptive User-Agent on API requests, so set one here
USER_AGENT = "NLP-Project10-TriviaQA-OpenQA/1.0 (contact: just2aritra@gmail.com)"

print("Config OK. Seed:", SEED, "| N_SAMPLE:", N_SAMPLE)

Config OK. Seed: 42 | N_SAMPLE: 300


In [2]:
# NLTK resources (sentence tokenizer, POS tagger) used for sentence splitting
# and noun-phrase extraction in the heuristic answer extractor.
# Downloading only the ones that are missing, quietly, so re-running this cell
# does not re-download anything that is already cached
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find(f"taggers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag

# Quick smoke test to make sure the sentence tokenizer actually loaded fine
print("NLTK ready:", sent_tokenize("A test sentence. Another one."))

NLTK ready: ['A test sentence.', 'Another one.']


## 2. Load TriviaQA (`rc.nocontext`), sample a fixed question set

In [3]:
# Loading the TriviaQA rc.nocontext validation split from HuggingFace
# (downloaded once, then cached locally after that)
raw_ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
print(raw_ds)
print(raw_ds.features)

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Dataset({
    features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
    num_rows: 17944
})
{'question': Value('string'), 'question_id': Value('string'), 'question_source': Value('string'), 'entity_pages': {'doc_source': List(Value('string')), 'filename': List(Value('string')), 'title': List(Value('string')), 'wiki_context': List(Value('string'))}, 'search_results': {'description': List(Value('string')), 'filename': List(Value('string')), 'rank': List(Value('int32')), 'title': List(Value('string')), 'url': List(Value('string')), 'search_context': List(Value('string'))}, 'answer': {'aliases': List(Value('string')), 'normalized_aliases': List(Value('string')), 'matched_wiki_entity_name': Value('string'), 'normalized_matched_wiki_entity_name': Value('string'), 'normalized_value': Value('string'), 'type': Value('string'), 'value': Value('string')}}


In [4]:
def flatten_gold_answers(answer_dict):
    """Combine answer['value'] and every entry in answer['aliases'] into one
    flat list of acceptable strings, as required by the spec (preprocessing step 2)."""

    # Initialising the list that will hold every acceptable gold string
    golds = []
    if answer_dict.get("value"):
        golds.append(answer_dict["value"])
    for a in answer_dict.get("aliases", []) or []:
        golds.append(a)

    # also fold in normalized_aliases / normalized_value for extra robustness
    if answer_dict.get("normalized_value"):
        golds.append(answer_dict["normalized_value"])
    for a in answer_dict.get("normalized_aliases", []) or []:
        golds.append(a)

    # dedupe, preserve order
    seen, out = set(), []
    for g in golds:
        if g not in seen:
            seen.add(g)
            out.append(g)
    return out


# Shuffling all validation indices with the fixed seed, then taking the first
# N_SAMPLE of them -- this is the fixed 300-question sample reused everywhere
rng = random.Random(SEED)
all_indices = list(range(len(raw_ds)))
rng.shuffle(all_indices)
sample_indices = sorted(all_indices[:N_SAMPLE])

# Building the sample: for each picked index, pulling out the question, the
# flattened gold answers, and a couple of metadata fields we will want later
samples = []
for idx in sample_indices:
    row = raw_ds[idx]
    gold_answers = flatten_gold_answers(row["answer"])
    samples.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "answer_value": row["answer"]["value"],
        "answer_type": row["answer"].get("type", ""),
        "gold_answers": gold_answers,
    })

print(f"Sampled {len(samples)} questions (seed={SEED}).")
samples[:3]

Sampled 300 questions (seed=42).


[{'question_id': 'tc_106',
  'question': 'Which actress was voted Miss Greenwich Village in 1942?',
  'answer_value': 'Lauren Bacall',
  'answer_type': 'WikipediaEntity',
  'gold_answers': ['Lauren Bacall',
   'Bacall',
   'Lauren Becal',
   'Lauren Becall',
   'Betty J. Perske',
   'Loren Bacall',
   'Betty Joan Perske',
   'Betty Perske',
   'Betty Joan Perski',
   'lauren bacall',
   'lauren becall',
   'loren bacall',
   'lauren becal',
   'betty j perske',
   'betty perske',
   'betty joan perske',
   'bacall',
   'betty joan perski']},
 {'question_id': 'tc_1331',
  'question': 'Who had a 60s No 1 hit with This Diamond Ring?',
  'answer_value': 'Gary Lewis and the Playboys',
  'answer_type': 'WikipediaEntity',
  'gold_answers': ['Gary Lewis and the Playboys',
   'Dave Gonzalez (Gary Lewis & the Playboys)',
   'Gary Lewis & the Playboys',
   'Gary Lewis and The Playboys',
   'Gary Lewis & The Playboys',
   'gary lewis and playboys',
   'dave gonzalez gary lewis playboys',
   'gary 

In [5]:
# Persist the sample so Tasks 2 and 3 reuse the exact same 300 questions.
with open(SAMPLE_PATH, "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")
print("Saved sample to", SAMPLE_PATH)

Saved sample to ../../data/triviaqa_sample_300.jsonl


## 3. Build Wikipedia-paragraph index from scratch (built once, reused in Tasks 2 & 3)

In [6]:
# Setting up the Wikipedia API endpoint used to resolve answer strings to
# real page titles and then fetch their plain-text extracts
WIKI_API = "https://en.wikipedia.org/w/api.php"


def _wiki_get(params, retries=3, timeout=15):
    """Utility function that hits the Wikipedia API with the given params and
    retries a few times (with backoff) before giving up and returning None."""

    params = {**params, "format": "json"}
    url = WIKI_API + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})

    # Iterating through the retry attempts, backing off a bit more each time
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8"))
        except Exception as e:
            if attempt == retries - 1:
                return None
            time.sleep(1.5 * (attempt + 1))
    return None


def resolve_wiki_title(query):
    """Resolve a free-text query (here: a TriviaQA answer string) to the best-matching
    Wikipedia page title via the opensearch API."""

    data = _wiki_get({"action": "opensearch", "search": query, "limit": 1, "namespace": 0})
    if data and len(data) >= 2 and data[1]:
        return data[1][0]
    return None


def fetch_wiki_extract(title):
    """Fetch the plain-text extract of a Wikipedia page (follows redirects)."""

    data = _wiki_get({
        "action": "query", "prop": "extracts", "explaintext": 1,
        "redirects": 1, "titles": title,
    })
    if not data:
        return None

    # Iterating through the returned pages dict to find the first valid page
    # (page_id "-1" means Wikipedia could not find the title at all)
    pages = data.get("query", {}).get("pages", {})
    for page_id, page in pages.items():
        if page_id == "-1" or "extract" not in page:
            continue
        return {"title": page.get("title", title), "extract": page["extract"]}
    return None


print("Wikipedia fetch helpers defined.")

Wikipedia fetch helpers defined.


In [7]:
def normalize_topic_query(s):
    """Deterministic cleanup applied uniformly to every answer string before searching
    Wikipedia -- never chooses between alternative words/substrings, only strips
    boilerplate that can't be part of a real article title."""

    s = s.strip()
    s = re.sub(r'["\'.!]+$', "", s)              # trailing punctuation/quotes
    s = re.sub(r"^(a|an|the)\s+", "", s, flags=re.IGNORECASE)  # leading article
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)        # trailing parenthetical
    s = s.strip()
    return s


# Quick smoke test to check the cleanup rules behave as expected
print(normalize_topic_query("A Hippopotamus."))
print(normalize_topic_query("The US Federal Reserve"))
print(normalize_topic_query('"MEASURE FOR MEASURE"'))

Hippopotamus
US Federal Reserve
"MEASURE FOR MEASURE


In [8]:
# Load cache of {answer_value -> page dict} if present, so re-running the notebook
# does not re-hit the network.
if WIKI_CACHE_PATH.exists():
    with open(WIKI_CACHE_PATH) as f:
        wiki_cache = json.load(f)
else:
    wiki_cache = {}

# Getting the unique set of topics (answer strings) we actually need to resolve
unique_topics = sorted({s["answer_value"] for s in samples if s["answer_value"]})
print(f"{len(unique_topics)} unique topics to resolve/fetch out of {len(samples)} questions.")

# Iterating through every unique topic and fetching its Wikipedia page,
# falling back through a few strategies before giving up on it
failed_topics = []
for topic in tqdm(unique_topics, desc="Fetching Wikipedia pages"):
    if topic in wiki_cache and wiki_cache[topic] is not None:
        continue

    resolved_title = resolve_wiki_title(topic)
    page = fetch_wiki_extract(resolved_title) if resolved_title else None
    if page is None:
        page = fetch_wiki_extract(topic)  # try the raw answer string directly

    if page is None:
        # fixed, deterministic normalization fallback (strip trailing punctuation,
        # leading article, trailing parenthetical) -- same rule for every topic,
        # not a per-topic choice among candidate words
        norm_topic = normalize_topic_query(topic)
        if norm_topic and norm_topic != topic:
            resolved_title = resolve_wiki_title(norm_topic)
            page = fetch_wiki_extract(resolved_title) if resolved_title else None

    wiki_cache[topic] = page
    if page is None:
        failed_topics.append(topic)
    time.sleep(0.05)  # be polite to the API

# Persisting the (possibly updated) cache back to disk
with open(WIKI_CACHE_PATH, "w") as f:
    json.dump(wiki_cache, f)

n_ok = sum(1 for v in wiki_cache.values() if v is not None)
print(f"Resolved {n_ok}/{len(unique_topics)} topics. Failed: {len(failed_topics)}")
if failed_topics:
    print("Examples of failed topics:", failed_topics[:10])

294 unique topics to resolve/fetch out of 300 questions.


Fetching Wikipedia pages:   0%|          | 0/294 [00:00<?, ?it/s]

Resolved 287/294 topics. Failed: 7
Examples of failed topics: ["'MEASURE FOR MEASURE'", 'A sports celebrity of the host country', 'Dr. (James) Parkinson', 'Karl Marx and Friedrich Engels', 'Per Annum Exclusive', 'The first space burial', 'Used furniture dealer']


In [9]:
MIN_PARA_CHARS = 80  # filter out very short / boilerplate paragraphs (section stubs, etc.)


def split_into_paragraphs(extract_text):
    """Utility function that splits a Wikipedia plaintext extract into clean
    paragraphs, dropping section headings and anything too short to be useful."""

    # Wikipedia plaintext extracts use "== Heading ==" lines for sections; drop those,
    # and split on blank lines for paragraphs.
    paras = []
    for block in extract_text.split("\n\n"):
        block = block.strip()
        if not block:
            continue
        if re.match(r"^=+.*=+$", block):  # section heading line
            continue
        block = re.sub(r"\s+", " ", block)
        if len(block) >= MIN_PARA_CHARS:
            paras.append(block)
    return paras


# Building the paragraph-level passage pool: one entry per paragraph, tagged
# with the Wikipedia page title it came from
passages = []  # list of dicts: passage_id, doc_title, text
seen_doc_titles = set()
for topic, page in wiki_cache.items():
    if page is None:
        continue
    title = page["title"]
    if title in seen_doc_titles:
        continue
    seen_doc_titles.add(title)
    for para in split_into_paragraphs(page["extract"]):
        passages.append({
            "passage_id": len(passages),
            "doc_title": title,
            "text": para,
        })

print(f"Pooled corpus: {len(seen_doc_titles)} unique Wikipedia pages -> {len(passages)} paragraphs.")

Pooled corpus: 284 unique Wikipedia pages -> 7660 paragraphs.


In [10]:
# Persist the index once, so Task 2 and Task 3 notebooks can load it directly
# instead of re-fetching from Wikipedia.
with open(INDEX_PATH, "w") as f:
    for p in passages:
        f.write(json.dumps(p) + "\n")
print("Saved paragraph index to", INDEX_PATH)

passages_df = pd.DataFrame(passages)
passages_df.head(3)

Saved paragraph index to ../../data/wiki_paragraph_index.jsonl


,passage_id,doc_title,text
0,0,Happy Ever After (British TV series),Happy Ever After is a British sitcom starring ...
1,1,Happy Ever After (British TV series),== Cast == Terry Scott as Terry Fletcher June ...
2,2,Happy Ever After (British TV series),== Plot == Terry and June Fletcher are a middl...


## 4. BM25 index (k1=1.5, b=0.75)

In [11]:
# Initialising the regex used to pull out alphanumeric tokens for BM25
TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text):
    """Utility function to lowercase and tokenize text for BM25 scoring."""
    return TOKEN_RE.findall(text.lower())


# Tokenizing every passage once, up front, so BM25Okapi can build its index
corpus_texts = [p["text"] for p in passages]
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenizing corpus")]

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("BM25 index built over", len(tokenized_corpus), "paragraphs.")

Tokenizing corpus:   0%|          | 0/7660 [00:00<?, ?it/s]

BM25 index built over 7660 paragraphs.


## 5. Retrieval: BM25 top-k for every question

In [12]:
def retrieve(question, top_k=TOP_K_RETRIEVE):
    """Utility function that scores every passage against the question with BM25
    and returns the top_k (passage_id, score) pairs, highest score first."""

    q_tokens = tokenize(question)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]


# Iterating through every sampled question and retrieving its top-k passages
retrieval_results = []
for s in tqdm(samples, desc="Retrieving"):
    hits = retrieve(s["question"], top_k=TOP_K_RETRIEVE)
    retrieval_results.append(hits)

print("Example retrieval for:", samples[0]["question"])
for pid, score in retrieval_results[0][:3]:
    print(f"  score={score:.2f} doc={passages[pid]['doc_title']!r} text={passages[pid]['text'][:120]!r}")

Retrieving:   0%|          | 0/300 [00:00<?, ?it/s]

Example retrieval for: Which actress was voted Miss Greenwich Village in 1942?
  score=26.61 doc='Lauren Bacall' text='In 1941, Bacall studied for a year at the American Academy of Dramatic Arts in New York, where she met classmate and fut'
  score=23.42 doc='Buddy Holly' text='==== Richard Barone ==== In 2016, Richard Barone released his album Sorrows & Promises: Greenwich Village in the 1960s, '
  score=23.25 doc='Fred Perry' text='=== 1942–1945 === After breaking his elbow in a match against Bobby Riggs on the opening night of the Round robin World '


## 6. Answering: no generation, lexical-overlap sentence -> longest NP/number span

In [13]:
# Loading the English stopwords list once, extended with a handful of question
# words (who/what/when/...) that would otherwise pollute the lexical overlap score
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english")) | {
    "who", "what", "when", "where", "which", "whom", "whose", "how", "many", "much"
}

# Base NP (no PP-attachment): a noun phrase should not swallow a trailing
# "... of X" / "... in Y" chain, or "longest span" degenerates into picking the
# longest unrelated prepositional tail instead of the actual entity/number.
CHUNK_GRAMMAR = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}
  NUM: {<CD>+}
"""
chunk_parser = nltk.RegexpParser(CHUNK_GRAMMAR)


def overlap_score(question, sentence):
    """Utility function that scores a sentence by how many non-stopword
    question terms it contains, normalized by the number of question terms."""

    q_words = set(tokenize(question)) - STOPWORDS
    s_words = set(tokenize(sentence)) - STOPWORDS
    if not q_words or not s_words:
        return 0.0
    return len(q_words & s_words) / len(q_words)


def best_sentence(question, passage_text):
    """Utility function that picks the sentence in passage_text with the
    highest lexical overlap against the question."""

    sentences = sent_tokenize(passage_text)
    if not sentences:
        return ""
    scored = [(overlap_score(question, s), s) for s in sentences]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def extract_span_candidates(sentence):
    """Utility function that POS-tags and chunks a sentence, returning every
    NP / numeric span found in it."""

    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)
    tree = chunk_parser.parse(tagged)

    # Iterating through the parsed chunk tree and collecting every NP/NUM span
    spans = []
    for subtree in tree.subtrees():
        if subtree.label() in ("NP", "NUM"):
            span_text = " ".join(w for w, t in subtree.leaves())
            spans.append(span_text)
    return spans


def extract_answer(question, passage_text):
    """Heuristic extractive answer: best sentence -> longest NP/number span."""

    sent = best_sentence(question, passage_text)
    if not sent:
        return ""
    candidates = extract_span_candidates(sent)
    if not candidates:
        return sent  # fallback: whole sentence if no chunk found
    return max(candidates, key=len)


# quick smoke test
demo_sent = "Jupiter is the fifth planet from the Sun and the largest in the Solar System."
print(extract_span_candidates(demo_sent))

['Jupiter', 'the fifth planet', 'the Sun', 'the Solar System']


In [14]:
# Iterating through every question, taking its top-1 retrieved passage, and
# running the heuristic extractor on it to get the final predicted answer
predictions = []
for s, hits in tqdm(list(zip(samples, retrieval_results)), desc="Extracting answers"):
    if not hits:
        predictions.append("")
        continue
    top1_pid, _ = hits[0]
    top1_text = passages[top1_pid]["text"]
    pred = extract_answer(s["question"], top1_text)
    predictions.append(pred)

print("Done. Example:")
print("Q:", samples[0]["question"])
print("Gold:", samples[0]["gold_answers"])
print("Pred:", predictions[0])

Extracting answers:   0%|          | 0/300 [00:00<?, ?it/s]

Done. Example:
Q: Which actress was voted Miss Greenwich Village in 1942?
Gold: ['Lauren Bacall', 'Bacall', 'Lauren Becal', 'Lauren Becall', 'Betty J. Perske', 'Loren Bacall', 'Betty Joan Perske', 'Betty Perske', 'Betty Joan Perski', 'lauren bacall', 'lauren becall', 'loren bacall', 'lauren becal', 'betty j perske', 'betty perske', 'betty joan perske', 'bacall', 'betty joan perski']
Pred: Miss Greenwich Village


## 7. Evaluation: EM, F1, recall@k

In [15]:
def normalize_answer(s):
    """SQuAD-style normalization: lowercase, remove punctuation, remove articles, fix whitespace."""

    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def compute_em(pred, golds):
    """Utility function to return 1.0 if the prediction exactly matches any
    gold answer (after normalization), else 0.0."""

    norm_pred = normalize_answer(pred)
    return float(any(norm_pred == normalize_answer(g) for g in golds))


def f1_score(pred, gold):
    """Utility function that computes token-level F1 between one prediction
    and one gold answer string."""

    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_f1(pred, golds):
    """Utility function that takes the best F1 across all acceptable gold answers."""
    return max(f1_score(pred, g) for g in golds) if golds else 0.0


def recall_at_k(hits, golds, k, passages_ref):
    """Utility function to check whether any gold answer string appears
    (as a normalized substring) in any of the top-k retrieved passages."""

    norm_golds = [normalize_answer(g) for g in golds]
    for pid, _ in hits[:k]:
        norm_text = normalize_answer(passages_ref[pid]["text"])
        if any(g and g in norm_text for g in norm_golds):
            return 1.0
    return 0.0


print("Metric functions defined.")

Metric functions defined.


In [16]:
# Iterating through every question and building one results record per
# question: gold answers, prediction, EM/F1, and recall@k for every k we track
records = []
for s, hits, pred in zip(samples, retrieval_results, predictions):
    golds = s["gold_answers"]
    rec = {
        "question_id": s["question_id"],
        "question": s["question"],
        "gold_answers": golds,
        "prediction": pred,
        "em": compute_em(pred, golds),
        "f1": compute_f1(pred, golds),
        "top1_doc": passages[hits[0][0]]["doc_title"] if hits else None,
    }
    for k in K_LIST:
        rec[f"recall@{k}"] = recall_at_k(hits, golds, k, passages)
    records.append(rec)

results_df = pd.DataFrame(records)
results_df.to_csv(RESULTS_DIR / "task1_per_question_results.csv", index=False)
results_df.head(5)

,question_id,question,gold_answers,prediction,em,f1,top1_doc,recall@1,recall@3,recall@5,recall@10
0,tc_106,Which actress was voted Miss Greenwich Village...,"[Lauren Bacall, Bacall, Lauren Becal, Lauren B...",Miss Greenwich Village,0.0,0.0,Lauren Bacall,1.0,1.0,1.0,1.0
1,tc_1331,Who had a 60s No 1 hit with This Diamond Ring?,"[Gary Lewis and the Playboys, Dave Gonzalez (G...",a recording studio,0.0,0.0,Gary Lewis & the Playboys,1.0,1.0,1.0,1.0
2,tc_2020,What percentage of the earth's surface is cove...,"[8%, eight %, 8 %, 8, eight]",Another interpretation,0.0,0.0,Antediluvian,0.0,1.0,1.0,1.0
3,tc_2167,Which US city was named after a British Prime ...,"[Pittsburgh, Smoky City, Pittsburgh (Pa.), Pit...",British prime minister,0.0,0.0,Margaret Thatcher,0.0,0.0,0.0,0.0
4,tc_2900,"Which teeth are the third molars, and are alwa...","[Wisdom teeth, Wisdom-tooth, Third maxillary m...",the human dentition,0.0,0.0,Wisdom tooth,1.0,1.0,1.0,1.0


In [17]:
# Averaging EM/F1/recall@k over all 300 questions and saving the summary metrics
metrics = {
    "n_questions": len(results_df),
    "EM": results_df["em"].mean(),
    "F1": results_df["f1"].mean(),
}
for k in K_LIST:
    metrics[f"recall@{k}"] = results_df[f"recall@{k}"].mean()

with open(RESULTS_DIR / "task1_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("=== Task 1 (TriviaQA rc.nocontext) — BM25 + heuristic extraction ===")
for key, val in metrics.items():
    print(f"{key:>12}: {val:.4f}" if isinstance(val, float) else f"{key:>12}: {val}")

=== Task 1 (TriviaQA rc.nocontext) — BM25 + heuristic extraction ===
 n_questions: 300
          EM: 0.0200
          F1: 0.0927
    recall@1: 0.5333
    recall@3: 0.6600
    recall@5: 0.7167
   recall@10: 0.7433


## 8. Qualitative analysis: correct vs. incorrect examples

In [18]:
def show_examples(df, n=5, correct=True, seed=SEED):
    """Utility function to print n random correct (EM=1) or incorrect (EM=0)
    examples from the results dataframe, for qualitative inspection."""

    subset = df[df["em"] == (1.0 if correct else 0.0)]
    subset = subset.sample(min(n, len(subset)), random_state=seed)

    # Iterating through the sampled rows and printing question/gold/pred side by side
    for _, row in subset.iterrows():
        print(f"Q: {row['question']}")
        print(f"   Gold: {row['gold_answers']}")
        print(f"   Pred: {row['prediction']!r}")
        print(f"   Top-1 doc: {row['top1_doc']}")
        print(f"   EM={row['em']}  F1={row['f1']:.2f}")
        print("-" * 80)


print(">>> CORRECT (EM=1) examples")
show_examples(results_df, n=5, correct=True)

print("\n>>> INCORRECT (EM=0) examples")
show_examples(results_df, n=5, correct=False)

>>> CORRECT (EM=1) examples
Q: By what name is comic book villain Oswald Chesterfield Cobblepot better known?
   Gold: ['The Penguin', 'Penguin (comics)', 'The Penguin (comics)', 'Oswald Chesterfield Cobblepot', 'Actors to portray the Penguin', 'Cobblepot', 'Oswald Cobblepot', 'The penguin', 'Penguin (Oswald Chesterfield Cobblepot)', 'Penguin (character)', 'penguin', 'actors to portray penguin', 'penguin comics', 'oswald cobblepot', 'oswald chesterfield cobblepot', 'penguin oswald chesterfield cobblepot', 'cobblepot', 'penguin character']
   Pred: 'Oswald Chesterfield Cobblepot'
   Top-1 doc: Penguin (character)
   EM=1.0  F1=1.00
--------------------------------------------------------------------------------
Q: The Marquesas Islands lie in which ocean?
   Gold: ['Pacific', 'Pacific Basin', 'Oceanum pacificum', 'Sea of Magellan', 'Pacific region', 'PAcific', 'North Pacific Ocean', 'Peaceful sea', 'Eastern Pacific Ocean', 'Mar del Sur', 'South-Western Pacific', 'West Pacific', 'Eastern

## 9. Summary

EM=0.020, F1=0.093, recall@1=0.53, recall@5=0.72, recall@10=0.74 (n=300). Full discussion and
limitations: see report.